# 상관 특성 섭동을 포함한 LinearExplainer의 수학

`LinearExplainer(model,prior,feature_perturbation="correlation_dependent")`를 사용할 때 $E[f(x) \mid do(X_S = x_S)]$를 사용하여 특성 세트 $S$의 영향을 측정하지 않고 대신 무작위 변수 $X$(입력 특성을 나타냄)가 다변량 가시안을 따른다는 가정 하에 $E[f(x) \mid X_S = x_s]$를 사용합니다. 유통. 이러한 방식으로 SHAP 값을 계산하려면 모든 기능 하위 집합에 대한 다변량 가시안 분포에서 조건부 기대치를 계산해야 합니다. 이는 기하급수적인 수의 용어에 대해 많은 행렬 일치가 발생하므로 몇 가지 기능 이상을 가진 모델에서는 다루기 어렵습니다.

이 문서에서는 한 번만 수행한 다음 원하는 만큼 많은 샘플에 적용할 수 있는 샘플링 절차를 사용하여 필요한 모든 선형 대수학을 미리 계산하는 데 사용한 수학에 대해 간략하게 설명합니다. 이는 무차별 접근 방식에 비해 계산 속도를 대폭 향상시킵니다. 이 모든 계산은 선형 모델 $f(x) = \beta x$를 설명한다는 사실에 의존합니다.

대부분의 설명자가 사용하는 개입 형태의 SHAP 값의 순열 정의는 다음과 같습니다.

$$
\phi_i = \frac{1}{M!} \sum_R E[f(X) \mid do(X_{S_i^R \cup i} = x_{S_i^R \cup i})] - E[f(X) \mid do(X_{S_i^R} = x_{S_i^R})]
$$

그러나 여기서는 비개입 조건부 기대 형식(확률 변수 $X$에 대한 명시적 참조를 삭제하여 표기법을 단순화함)을 사용합니다.

$$
\phi_i = \frac{1}{M!} \sum_R E[f(x) \mid x_{S_i^R \cup i}] - E[f(x) \mid x_{S_i^R}]
$$

여기서 $f(x) = \beta x + b$는 $\beta$가 행 벡터이고 $b$가 스칼라입니다.

f(x)를 선형 함수 정의로 바꾸면 다음을 얻습니다.

\begin{정렬}
\phi_i = \frac{1}{M!} \sum_R E[\beta x + b \mid x_{S_i^R \cup i}] - E[\beta x + b \mid x_{S_i^R}] \\
 = \beta \frac{1}{M!} \sum_R E[x \mid x_{S_i^R \cup i}] - E[x \mid x_{S_i^R}]
\end{정렬}

입력 $x$가 평균 $\mu$ 및 공분산 $\Sigma$를 갖는 다변량 정규 분포를 따른다고 가정합니다. 세트 $S$를 $P_S$로 선택하는 투영 행렬을 표시하면 다음을 얻습니다.

\begin{정렬}
E[x \mid x_S] = [P_{\bar S} \mu + P_{\bar S} \시그마 P_S^T (P_S \시그마 P_S^T)^{-1} ( P_S x - P_S \mu)] P_{\bar S} + x P_S^T P_S \\
= [P_{\bar S} \mu + P_{\bar S} \시그마 P_S^T (P_S \시그마 P_S^T)^{-1} P_S (x - \mu)] P_{\bar S} + x P_S^T P_S \\
= [\mu + \시그마 P_S^T (P_S \시그마 P_S^T)^{-1} P_S (x - \mu)] P_{\bar S}^T P_{\bar S} + x P_S^T P_S \\
= P_{\bar S}^T P_{\bar S} [\mu + \Sigma P_S^T (P_S \Sigma P_S^T)^{-1} P_S (x - \mu)] + P_S^T P_S x \\
= P_{\bar S}^T P_{\bar S} \mu + P_{\bar S}^T P_{\bar S} \시그마 P_S^T (P_S \Sigma P_S^T)^{-1} P_S x - P_{\bar S}^T P_{\bar S} \Sigma P_S^T (P_S \Sigma P_S^T)^{-1} P_S \mu + P_S^T P_S x \\
= [P_{\bar S}^T P_{\bar S} - P_{\bar S}^T P_{\bar S} \시그마 P_S^T (P_S \시그마 P_S^T)^{-1} P_S] \mu + [P_S^T P_S + P_{\bar S}^T P_{\bar S} \시그마 P_S^T (P_S \Sigma P_S^T)^{-1} P_S] x
\end{정렬}

$R_S = P_{\bar S}^T P_{\bar S} \Sigma P_S^T (P_S \Sigma P_S^T)^{-1} P_S$ 및 $Q_S = P_S^T P_S$라고 하면 다음과 같이 쓸 수 있습니다.

\begin{정렬}
E[x \mid x_S] = [Q_{\bar S} - R_S] \mu + [Q_S + R_S] x
\end{정렬}

또는

\begin{정렬}
E[x \mid x_{S_i^R \cup i}] = [Q_{\bar{S_i^R \cup i}} - R_{S_i^R \cup i}] \mu + [Q_{S_i^R \cup i} + R_{S_i^R \cup i}] x
\end{정렬}

Shapley 방정식으로 이어지는

\begin{정렬}
\phi_i = \beta \frac{1}{M!} \sum_R [Q_{\bar{S_i^R \cup i}} - R_{S_i^R \cup i}] \mu + [Q_{S_i^R \cup i} + R_{S_i^R \cup i}] x - [Q_{\bar{S_i^R}} - R_{S_i^R}] \mu - [Q_{S_i^R} + R_{S_i^R}] x \\
= \beta \frac{1}{M!} \sum_R ([Q_{\bar{S_i^R \cup i}} - R_{S_i^R \cup i}] - [Q_{\bar{S_i^R}} - R_{S_i^R}]) \mu + ([Q_{S_i^R \cup i} + R_{S_i^R \cup i}] - [Q_{S_i^R} + R_{S_i^R}]) x \\
= \beta \left [ \frac{1}{M!} \sum_R ([Q_{\bar{S_i^R \cup i}} - R_{S_i^R \cup i}] - [Q_{\bar{S_i^R}} - R_{S_i^R}]) \right ] \mu + \beta \left [ \frac{1}{M!} \sum_R ([Q_{S_i^R \cup i} + R_{S_i^R \cup i}] - [Q_{S_i^R} + R_{S_i^R}]) \right ] x
\end{정렬}




$$
\phi = \beta T x
$$

이는 무작위 순열 $R$를 여러 번 추출하고 결과를 평균화하여 변환 행렬 $T$를 미리 계산할 수 있음을 의미합니다. $T$를 계산한 후에는 행렬 곱셈을 사용하여 원하는 수의 샘플(또는 해당 문제에 대한 모델)을 설명할 수 있습니다.